In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os
import seaborn as sns
from scipy import signal

class DroneDataLoader:
    def __init__(self, data_path='..'):
        self.data_path = data_path
        self.index_path = os.path.join(data_path, 'data/dataset_index.csv')
        if not os.path.exists(self.index_path):
            print(f"Index file not found at {self.index_path}")
            return
        
        self.df = pd.read_csv(self.index_path)
        print(f"Registry successfully loaded. Available records: {len(self.df)}")

    def get_samples(self, category=None, freq=None):
        filtered = self.df
        if category:
            filtered = filtered[filtered['category'] == category]
        if freq:
            filtered = filtered[filtered['freq_ghz'] == freq]
        return filtered
    
    def load_data_pair(self, row):
        vtx_path = os.path.join(self.data_path, row['csv_vtx_path'].lstrip('./')) if pd.notna(row['csv_vtx_path']) else None
        vrx_path = os.path.join(self.data_path, row['csv_vrx_path'].lstrip('./'))
        
        vtx = pd.read_csv(vtx_path, skiprows=1) if vtx_path else None
        vrx = pd.read_csv(vrx_path, skiprows=1)
        
        return vtx, vrx
    
    def plot_comparative_analysis(self, row, start_time_s = 0, end_time_s=0.2):
        vtx, vrx = self.load_data_pair(row)
        vtx = self.normalize_signal(vtx)
        vrx = self.normalize_signal(vrx, window_size=7_000)
        
        mask_time = (vrx['Second'] >= start_time_s) & (vrx['Second'] <= end_time_s)
        
        fig = plt.figure(figsize=(18, 6))
        gs = fig.add_gridspec(1, 2, width_ratios=[3, 2])
        
        ax1 = fig.add_subplot(gs[0, 0])
        
        ax1.plot(vrx[mask_time]['Second'] * 10e3, vrx[mask_time]['Value'], 
                 label='VRX (Received)', color='tab:blue', alpha=0.4, lw=1)
        
        if vtx is not None:
            ax1.plot(vtx[mask_time]['Second'] * 10e3, vtx[mask_time]['Value'], 
                     label='VTX (Original)', color='tab:green', alpha=0.4, lw=1)
        
        ax1.set_title(f"Signal Comparison: Sample {row['idx']} ({row['freq_ghz']} GHz)")
        ax1.set_xlabel("Time (ms)")
        ax1.set_ylabel("Normalized Amplitude")
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        ax2 = fig.add_subplot(gs[0, 1])
        img_path = os.path.join(self.data_path, row['img_path'].lstrip('./'))
        if os.path.exists(img_path):
            img = mpimg.imread(img_path)
            ax2.imshow(img)
            ax2.set_title("Visual Confirmation")
            ax2.axis('off')
        else:
            ax2.text(0.5, 0.5, "Image not found", ha='center', va='center')
            
        plt.tight_layout()
        plt.show()

    @staticmethod
    def normalize_signal(df, window_size=1_000_000):
        if df is None: return None
        data = df['Value'].values
        normalized = np.zeros_like(data)
    
        for i in range(0, len(data), window_size):
            end = min(i + window_size, len(data))
            window = data[i:end]

            normalized[i:end] = (window - np.min(window)) / (np.max(window) - np.min(window))

        new_df = df.copy()
        new_df['Value'] = normalized
        return new_df

    @staticmethod
    def calculate_snr(vtx_df, vrx_df):
        if vtx_df is None or vrx_df is None:
            return None

        s_ideal = vtx_df['Value'].values
        s_noisy = vrx_df['Value'].values

        noise = s_noisy - s_ideal
        
        p_signal = np.var(s_ideal)
        p_noise = np.var(noise)

        if p_noise == 0:
            return float('inf')

        snr_db = 10 * np.log10(p_signal / p_noise)
        return snr_db


loader = DroneDataLoader(data_path='..')

Registry successfully loaded. Available records: 104


# Methods
---
## Autocorrelation

In [2]:
def compute_acf(signal_df):
    signal_values = signal_df['Value'].values
    signal_values = signal_values - np.mean(signal_values)

    acf_vals = signal.correlate(signal_values, signal_values, mode='full', method='auto')
    acf_vals = acf_vals[acf_vals.size // 2:]
    if acf_vals[0] != 0:
        acf_vals /= acf_vals[0]
    
    lags = np.arange(len(acf_vals)) * (signal_df['Second'].iloc[1] - signal_df['Second'].iloc[0])
    
    return lags, acf_vals

def autocorr_sample(vrx_sample, mask_sync, mask_bg):
    acf_vals = compute_acf(vrx_sample)
    
    if acf_vals is None or len(acf_vals) == 0:
        return 0.0
    
    if not any(mask_sync) or not any(mask_bg):
        return 0.0
        
    max_sync = np.max(acf_vals[mask_sync])
    mean_sync = np.mean(acf_vals[mask_sync])
    
    avg_bg = np.mean(np.abs(acf_vals[mask_bg]))
    
    epsilon = 1e-6
    return (max_sync / (avg_bg + epsilon)), (mean_sync / (avg_bg + epsilon))

def autocorr_process(vrx_sample, sync_offsets_us, bg_offsets_us):
    lags = np.arange(len(vrx_sample) // 2) * (vrx_sample['Second'].iloc[1] - vrx_sample['Second'].iloc[0])
    
    sync_lags_s = np.asarray(sync_offsets_us, dtype=float) * 1e-6
    bg_lags_s = np.asarray(bg_offsets_us, dtype=float) * 1e-6

    mask_sync = np.zeros_like(lags, dtype=bool)
    mask_bg = np.zeros_like(lags, dtype=bool)

    if sync_lags_s.size:
        idx_sync = np.argmin(np.abs(lags[:, None] - sync_lags_s[None, :]), axis=0)
        mask_sync[idx_sync] = True

    if bg_lags_s.size:
        idx_bg = np.argmin(np.abs(lags[:, None] - bg_lags_s[None, :]), axis=0)
        mask_bg[idx_bg] = True
    
    if not any(mask_sync) or not any(mask_bg):
        return False
    
    max_ratio, mean_ratio = autocorr_sample(vrx_sample, mask_sync, mask_bg)
    return (max_ratio > 3.0) | (mean_ratio > 1.65)
        
    

---
## Matched filtering

In [ ]:
def compute_matched_filtering(vrx_sample, perfect_sample):
    if vrx_sample is None or perfect_sample is None:
        return None

    vrx_values = vrx_sample['Value'].values
    perfect_values = perfect_sample['Value'].values

    if len(vrx_values) != len(perfect_values):
        min_len = min(len(vrx_values), len(perfect_values))
        vrx_values = vrx_values[:min_len]
        perfect_values = perfect_values[:min_len]

    correlation = signal.correlate(vrx_values, perfect_values, mode='full')
    if (correlation[0] != 0):
        correlation /= correlation[0]

    lags = signal.correlation_lags(len(vrx_values), len(perfect_values), mode='full') * (vrx_sample['Second'].iloc[1] - vrx_sample['Second'].iloc[0])
    return lags, correlation

def matched_filtering_sample(vrx_sample, perfect_sample):
    result = compute_matched_filtering(vrx_sample, perfect_sample)
    if result is None:
        return 0.0
    
    _lags, correlation = result
    max_corr = np.max(correlation)
    mean_corr = np.mean(correlation)
    
    return max_corr / mean_corr if mean_corr != 0 else 0.0

def matched_filtering_process(vrx_sample, perfect_sample, threshold=5.0):
    ratio = matched_filtering_sample(vrx_sample, perfect_sample)
    return ratio > threshold

---
## FFT

In [5]:
def compute_ftt(signal_df):
    signal_values = signal_df['Value'].values
    signal_values = signal_values - np.mean(signal_values)

    freqs, psd = signal.welch(signal_values, fs=1/(signal_df['Second'].iloc[1] - signal_df['Second'].iloc[0]), nperseg=1024)
    
    return freqs, psd

def fft_sample(vrx_sample, mask_sync, mask_bg):
    freqs, psd = compute_ftt(vrx_sample)
    
    if freqs is None or psd is None:
        return 0.0
    
    max_sync = np.max(psd[mask_sync])
    mean_sync = np.mean(psd[mask_sync])
    
    avg_bg = np.mean(np.abs(psd[mask_bg]))
    
    epsilon = 1e-6
    return (max_sync / (avg_bg + epsilon)), (mean_sync / (avg_bg + epsilon))

def fft_process(vrx_sample, sync_freqs_ghz, bg_freqs_ghz):
    freqs, _psd = compute_ftt(vrx_sample)
    
    sync_freqs_hz = np.asarray(sync_freqs_ghz, dtype=float) * 1e9
    bg_freqs_hz = np.asarray(bg_freqs_ghz, dtype=float) * 1e9

    mask_sync = np.zeros_like(freqs, dtype=bool)
    mask_bg = np.zeros_like(freqs, dtype=bool)

    if sync_freqs_hz.size:
        idx_sync = np.argmin(np.abs(freqs[:, None] - sync_freqs_hz[None, :]), axis=0)
        mask_sync[idx_sync] = True

    if bg_freqs_hz.size:
        idx_bg = np.argmin(np.abs(freqs[:, None] - bg_freqs_hz[None, :]), axis=0)
        mask_bg[idx_bg] = True
    
    if not any(mask_sync) or not any(mask_bg):
        return False
    
    max_ratio, mean_ratio = fft_sample(vrx_sample, mask_sync, mask_bg)
    return (max_ratio > 1.5) | (mean_ratio > 1.5)

---
## Sync counting

In [ ]:
def sync_count_process(vrx_sample,
                       threshold_percent=5):
    sample_rate = vrx_sample['Second'].iloc[1] - vrx_sample['Second'].iloc[0]
    
    vals = vrx_sample['Value'].values.astype(np.int32)

    mn, mx = vals.min(), vals.max()
    range_ = mx - mn

    low_percentile = np.percentile(vals, 5)
    thr = low_percentile + (range_ * threshold_percent) // 100

    low_mask = vals <= thr
    low_edges = np.nonzero(np.logical_and(low_mask, np.concatenate(([True], ~low_mask[:-1]))))[0]

    expected_dist = (64*10e-6) / sample_rate 
    tol_percent = 0.2
    if len(low_edges) < 2:
        sync_pulses = 0
    else:
        delays = np.diff(low_edges)
        tolerance = max(1.0, expected_dist * 2 * tol_percent)
        sync_pulses = int(np.count_nonzero(np.abs(delays - expected_dist * (1 - tol_percent)) <= tolerance))

    expected_lines = len(vals) / expected_dist

    return sync_pulses/expected_lines